# Scopus API による Toyota研究者テーブルの精度検証（パイロット探索）

`toyota_data/toyota_authors.csv`（OpenAlex由来、9,429名）の所属判定・業績指標をScopus APIと突き合わせ、
どのエンドポイント・クエリ設計が実際に機能するかを少人数（既知ノイズ4名＋現所属ランダム4名の計8名）で試す。
WoS Starter APIキーは承認待ちのため、今回はScopus側のみを対象にする。

試すエンドポイント:
- Affiliation Search（`Toyota Motor`関連のaf-idを特定）
- Author Search（著者名＋AFFIL()での検索）
- Author Retrieval（Author Searchで見つかった候補IDの詳細取得）
- Scopus Search（AF-ID()を使った絞り込みクエリ）
- Abstract Retrieval（OpenAlex Works APIで取得した実際のDOIでの照合）

注意: `SCOPUS_API_KEY`はローカル環境変数からのみ読み、ノートブック・出力・リポジトリには一切書かない。
クォータ消費・レスポンス時間・エラー（429/403等）も記録し、本格展開時の見積もりに使う。

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
from urllib.parse import quote
import json
import os
import time

import pandas as pd
import requests

# 実行場所に依存せず、リポジトリ直下を探索する。
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'AGENTS.md').exists() and (candidate / 'data').is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError('リポジトリ直下を特定できません。リポジトリ内で実行してください。')

OUTPUT_DIR = ROOT / 'data' / 'derived' / 'scopus_pilot_exploration'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCOPUS_API_KEY = os.environ.get('SCOPUS_API_KEY')
if not SCOPUS_API_KEY:
    raise RuntimeError('SCOPUS_API_KEY が環境変数に設定されていません。')

SCOPUS_HEADERS = {'X-ELS-APIKey': SCOPUS_API_KEY, 'Accept': 'application/json'}
OPENALEX_HEADERS = {'User-Agent': 'mailto:r.ikura@msp-lab.org'}

RUN_AT_UTC = datetime.now(timezone.utc).isoformat()
print(f'RUN_AT_UTC = {RUN_AT_UTC}')
print(f'OUTPUT_DIR = {OUTPUT_DIR.relative_to(ROOT)}')


def scopus_get(source, url, params=None):
    """Scopus APIを1回呼び、状態・所要時間・レート制限残数・応答本体を記録する。キー自体は保存しない。"""
    started = perf_counter()
    try:
        resp = requests.get(url, headers=SCOPUS_HEADERS, params=params, timeout=30)
        duration_ms = round((perf_counter() - started) * 1000)
        remaining = resp.headers.get('X-RateLimit-Remaining')
        limit = resp.headers.get('X-RateLimit-Limit')
        record = {
            'source': source,
            'http_status': resp.status_code,
            'duration_ms': duration_ms,
            'rate_limit_remaining': remaining,
            'rate_limit_limit': limit,
            'url': resp.url.split('?')[0],
        }
        try:
            payload = resp.json()
        except ValueError:
            payload = None
        return record, payload
    except requests.RequestException as exc:
        return {
            'source': source,
            'http_status': None,
            'duration_ms': round((perf_counter() - started) * 1000),
            'rate_limit_remaining': None,
            'rate_limit_limit': None,
            'url': url,
            'error': str(exc),
        }, None


all_call_records = []

RUN_AT_UTC = 2026-07-27T01:35:01.188931+00:00
OUTPUT_DIR = data/derived/scopus_pilot_exploration


## 1. サンプル選定

`toyota_authors_README.md`に記載された既知ノイズ条件（`first_pub_year<1950` / `n_affiliations>100` / `n_last_known_institutions>10`）に該当する集合から4名、
それ以外で`is_currently_affiliated=True`の集合からランダムに4名を選ぶ（乱数シード固定で再現可能にする）。

In [2]:
df = pd.read_csv(ROOT / 'toyota_data' / 'toyota_authors.csv')

noise_mask = (df['first_pub_year'] < 1950) | (df['n_affiliations'] > 100) | (df['n_last_known_institutions'] > 10)
noise_subset = df[noise_mask]
clean_affiliated_subset = df[(df['is_currently_affiliated'] == True) & (~noise_mask)]

print(f'既知ノイズ該当: {len(noise_subset)}名')
print(f'現所属かつノイズ非該当: {len(clean_affiliated_subset)}名')

SEED = 20260727
sample_noise = noise_subset.sample(n=4, random_state=SEED)
sample_clean = clean_affiliated_subset.sample(n=4, random_state=SEED)

sample = pd.concat([
    sample_noise.assign(sample_group='known_noise'),
    sample_clean.assign(sample_group='currently_affiliated'),
], ignore_index=True)

sample_cols = ['sample_group', 'openalex_id', 'name', 'orcid', 'is_currently_affiliated',
               'matched_entities', 'works_count', 'cited_by_count', 'first_pub_year', 'n_affiliations']
sample[sample_cols]

既知ノイズ該当: 387名
現所属かつノイズ非該当: 3728名


,sample_group,openalex_id,name,orcid,is_currently_affiliated,matched_entities,works_count,cited_by_count,first_pub_year,n_affiliations
0,known_noise,A5112713997,Tetsuo Yazawa,NaN,False,Toyota Motor Corporation (Japan),266,3587,1936,20
1,known_noise,A5010897098,Masatoshi Ishikawa,0000-0002-6096-830X,False,Toyota Motor Corporation (Japan),1211,11429,1942,77
2,known_noise,A5108227345,Shin Yamamoto,NaN,False,Toyota Motor Corporation (Switzerland),462,5267,1935,69
3,known_noise,A5112074015,Minoru KAWAMOTO,NaN,False,Toyota Motor Corporation (Switzerland),202,260,1936,12
4,currently_affiliated,A5074188384,Tomohiro MIYABE,NaN,True,Toyota Motor Corporation (Switzerland),10,61,2002,3
5,currently_affiliated,A5059263401,Dorothée Lahaussois,NaN,True,Toyota Motor Corporation (Belgium),1,5,2022,1
6,currently_affiliated,A5087868824,Masaki Nobuhiro,NaN,True,Toyota Motor Corporation (Switzerland); Toyota...,2,2,2022,3
7,currently_affiliated,A5123709876,Masahiro Doi,NaN,True,Toyota Motor Corporation (Japan),1,1,2026,1


## 2. Affiliation Search

`Toyota Motor`関連のaf-id候補をScopus Affiliation Searchで特定する。後続のAF-ID()を使った絞り込みクエリに使う。

In [3]:
affil_record, affil_payload = scopus_get(
    'Affiliation Search: Toyota Motor',
    'https://api.elsevier.com/content/search/affiliation',
    params={'query': 'AFFIL(Toyota Motor)', 'count': 25},
)
all_call_records.append(affil_record)
print(affil_record)

affiliation_candidates = []
if affil_payload:
    entries = affil_payload.get('search-results', {}).get('entry', [])
    for e in entries:
        affiliation_candidates.append({
            'af_id': e.get('dc:identifier', '').replace('AFFILIATION_ID:', ''),
            'affiliation_name': e.get('affiliation-name'),
            'city': e.get('affiliation-city'),
            'country': e.get('affiliation-country'),
            'document_count': e.get('document-count'),
        })

affiliation_df = pd.DataFrame(affiliation_candidates)
affiliation_df

{'source': 'Affiliation Search: Toyota Motor', 'http_status': 200, 'duration_ms': 698, 'rate_limit_remaining': '4999', 'rate_limit_limit': '5000', 'url': 'https://api.elsevier.com/content/search/affiliation'}


,af_id,affiliation_name,city,country,document_count
0,60002347,Toyota Motor Corporation,None,None,5806
1,60309360,"Toyota Motor North America, Inc.",None,None,683
2,60340178,Toyota Motor Europe NV/SA,None,None,495
3,119829606,Toyota Motor North America,None,None,23
4,60356648,Toyota Motor North America Research & Development,None,None,18
5,123470102,Toyota Motor North America,None,None,17
6,114179704,Toyota Motor North America Inc.,None,None,16
7,100528980,Toyota Motor North America,None,None,15
8,122928765,Toyota Motor North America,None,None,13
9,125776629,Toyota Motor North America Inc.,None,None,11


## 3. Author Search

**重要な訂正**: Author Search APIは`AUTHOR-NAME()`という複合フィールドを受け付けない（`INVALID_INPUT`エラーになる）。
正しくは`AUTHLASTNAME()`と`AUTHFIRST()`をANDで組み合わせる（[Scopus Author Search Guide](https://nonprod-devportal.elsevier.com/sc_author_search_tips.html)で確認済み）。
また、OpenAlexの`name`列は姓名の順序が保証されないため、末尾トークンを姓とする素朴な分割を基本としつつ、0件のときは姓名を入れ替えて再試行する。
`AFFIL(Toyota)`は著者の**現在の所属だけでなく過去の全論文の所属表記**にもマッチする点に注意（`affiliation-current`と突き合わせて現在所属か過去所属かを区別する）。

In [4]:
def split_name(name):
    tokens = name.split()
    if len(tokens) < 2:
        return tokens[0], tokens[0]
    return tokens[-1], ' '.join(tokens[:-1])  # (last, first)


def author_search_once(last, first, affil_hint=None):
    query = f'AUTHLASTNAME({last}) AND AUTHFIRST({first})'
    if affil_hint:
        query += f' AND AFFIL({affil_hint})'
    record, payload = scopus_get(f'Author Search: {last},{first}' + (f'+{affil_hint}' if affil_hint else ''),
                                  'https://api.elsevier.com/content/search/author',
                                  params={'query': query, 'count': 5})
    all_call_records.append(record)
    candidates = []
    if payload:
        entries = payload.get('search-results', {}).get('entry', [])
        for e in entries:
            if 'error' in e:
                continue
            pref = e.get('preferred-name', {})
            affil_cur = e.get('affiliation-current') or {}
            candidates.append({
                'scopus_author_id': e.get('dc:identifier', '').replace('AUTHOR_ID:', ''),
                'preferred_name': f"{pref.get('surname','')}, {pref.get('given-name','')}",
                'affiliation_current': affil_cur.get('affiliation-name'),
                'affiliation_current_is_toyota': 'toyota' in (affil_cur.get('affiliation-name') or '').lower(),
                'document_count': e.get('document-count'),
            })
    return candidates


def author_search(name):
    """末尾トークンを姓とみなしAFFIL(Toyota)付きで検索。0件なら姓名を入れ替えて再試行する。"""
    last, first = split_name(name)
    candidates = author_search_once(last, first, 'Toyota')
    order_used = 'last-token-as-surname'
    if not candidates:
        candidates = author_search_once(first, last, 'Toyota')
        order_used = 'swapped'
    return candidates, order_used

author_search_results = {}
for _, row in sample.iterrows():
    candidates, order_used = author_search(row['name'])
    author_search_results[row['openalex_id']] = {'candidates': candidates, 'name_order_used': order_used}
    time.sleep(0.3)  # レート制限（5 req/秒）への配慮

for oid, res in author_search_results.items():
    name = sample.loc[sample['openalex_id'] == oid, 'name'].iloc[0]
    print(f"--- {name} ({oid}) : {len(res['candidates'])}件 (name_order={res['name_order_used']}) ---")
    for c in res['candidates']:
        print(' ', c)

--- Tetsuo Yazawa (A5112713997) : 1件 (name_order=last-token-as-surname) ---
  {'scopus_author_id': '7101659401', 'preferred_name': 'Yazawa, Tetsuo', 'affiliation_current': 'University of Hyogo', 'affiliation_current_is_toyota': False, 'document_count': '214'}
--- Masatoshi Ishikawa (A5010897098) : 0件 (name_order=swapped) ---
--- Shin Yamamoto (A5108227345) : 4件 (name_order=last-token-as-surname) ---
  {'scopus_author_id': '7408532252', 'preferred_name': 'Yamamoto, Shinichiro', 'affiliation_current': 'Aichi Prefectural University', 'affiliation_current_is_toyota': False, 'document_count': '167'}
  {'scopus_author_id': '7408531895', 'preferred_name': 'Yamamoto, Shin', 'affiliation_current': 'Meijo University', 'affiliation_current_is_toyota': False, 'document_count': '108'}
  {'scopus_author_id': '55475509500', 'preferred_name': 'Yamamoto, Shinichi', 'affiliation_current': 'Kariya Toyota General Hospital', 'affiliation_current_is_toyota': True, 'document_count': '3'}
  {'scopus_author_id

## 4. Author Retrieval

Author Searchで候補が1件以上見つかった場合、上位候補のScopus Author IDでAuthor Retrievalを呼び、詳細プロファイル（所属歴・主題分野・文献数・被引用数）を取得する。
被引用数はOpenAlexの値と単純比較・合算せず、別列として記録するだけにとどめる。

In [5]:
def author_retrieval(scopus_author_id):
    record, payload = scopus_get(f'Author Retrieval: {scopus_author_id}',
                                  f'https://api.elsevier.com/content/author/author_id/{scopus_author_id}')
    all_call_records.append(record)
    detail = {}
    if payload:
        core = payload.get('author-retrieval-response', [{}])
        if isinstance(core, list) and core:
            core = core[0]
        coredata = core.get('coredata', {}) if isinstance(core, dict) else {}
        detail = {
            'scopus_document_count': coredata.get('document-count'),
            'scopus_cited_by_count': coredata.get('cited-by-count'),
            'scopus_citation_count': coredata.get('citation-count'),
        }
    return record, detail

retrieval_results = {}
for oid, res in author_search_results.items():
    cands = res['candidates']
    if not cands:
        continue
    top = cands[0]
    record, detail = author_retrieval(top['scopus_author_id'])
    retrieval_results[oid] = {**top, **detail}
    time.sleep(0.3)

pd.DataFrame(retrieval_results).T if retrieval_results else '候補が見つかった著者なし'

,scopus_author_id,preferred_name,affiliation_current,affiliation_current_is_toyota,document_count,scopus_document_count,scopus_cited_by_count,scopus_citation_count
A5112713997,7101659401,"Yazawa, Tetsuo",University of Hyogo,False,214,214,2981,3621
A5108227345,7408532252,"Yamamoto, Shinichiro",Aichi Prefectural University,False,167,167,1103,1258
A5074188384,57200241226,"Miyabe, Tomohiro","Toyota Central R&D Labs., Inc.",True,1,1,4,4
A5059263401,57463625400,"Lahaussois, Dorothée",Toyota Motor Europe NV/SA,True,1,1,5,5
A5087868824,24335892200,"Nobuhiro, Masaki",IDEC Corporation,False,7,7,12,12
A5123709876,7201918691,"Doi, Masahiro",Toyota Motor Corporation,True,28,28,193,278


## 5. Scopus Search（AF-ID()による絞り込み vs AFFIL()の自由記述）

Scopus Search（`content/search/scopus`）は`AUTHOR-NAME()`フィールドを受け付ける（Author Searchとは別物）。
Affiliation Searchで得た主要`af-id`（文献数トップ3）で厳密に絞り込んだ場合と、`AFFIL(Toyota)`の自由記述文字列一致とで、ヒット件数がどう変わるかを比較する。
af-idはScopus側の機関表記ゆれ（部門・地域別に別af-idが立つ）を分けてしまうため、厳密な絞り込みほど取りこぼしが増える可能性がある。

In [6]:
search_comparison = {}
if not affiliation_df.empty:
    top_af_ids = affiliation_df.sort_values('document_count', ascending=False).head(3)['af_id'].tolist()
    af_id_clause = ' OR '.join(f'AF-ID({af})' for af in top_af_ids)

    for _, row in sample.iterrows():
        name = row['name']

        narrow_query = f"AUTHOR-NAME({name}) AND ({af_id_clause})"
        narrow_record, narrow_payload = scopus_get(f"Scopus Search narrow(AF-ID): {name}",
                                                     'https://api.elsevier.com/content/search/scopus',
                                                     params={'query': narrow_query, 'count': 5})
        all_call_records.append(narrow_record)

        broad_query = f'AUTHOR-NAME({name}) AND AFFIL(Toyota)'
        broad_record, broad_payload = scopus_get(f"Scopus Search broad(AFFIL): {name}",
                                                   'https://api.elsevier.com/content/search/scopus',
                                                   params={'query': broad_query, 'count': 5})
        all_call_records.append(broad_record)

        search_comparison[row['openalex_id']] = {
            'name': name,
            'narrow_af_id_hits': (narrow_payload or {}).get('search-results', {}).get('opensearch:totalResults'),
            'broad_affil_hits': (broad_payload or {}).get('search-results', {}).get('opensearch:totalResults'),
        }
        time.sleep(0.3)
else:
    print('af-id候補が見つからなかったため、このステップはスキップ')

search_comparison_df = pd.DataFrame(search_comparison).T
search_comparison_df

,name,narrow_af_id_hits,broad_affil_hits
A5112713997,Tetsuo Yazawa,0,0
A5010897098,Masatoshi Ishikawa,0,0
A5108227345,Shin Yamamoto,0,10
A5112074015,Minoru KAWAMOTO,0,0
A5074188384,Tomohiro MIYABE,0,0
A5059263401,Dorothée Lahaussois,0,0
A5087868824,Masaki Nobuhiro,0,4
A5123709876,Masahiro Doi,0,4


## 6. Abstract Retrieval（実DOIでの照合）

著者テーブルには論文単位のDOIがないため、OpenAlex Works APIでサンプルのうち2名分だけ代表的な論文DOIを取得し、
そのDOIをScopus Abstract Retrievalに照会して応答が得られるか・被引用数がどう出るかを確認する（クォータ節約のため2名に限定）。

In [7]:
def fetch_sample_doi(openalex_id):
    url = f'https://api.openalex.org/works'
    params = {'filter': f'authorships.author.id:{openalex_id}', 'sort': 'cited_by_count:desc', 'per_page': 1}
    resp = requests.get(url, params=params, headers=OPENALEX_HEADERS, timeout=30)
    resp.raise_for_status()
    results = resp.json().get('results', [])
    if not results:
        return None
    return results[0].get('doi', '').replace('https://doi.org/', '') if results[0].get('doi') else None


abstract_results = {}
for oid in sample['openalex_id'].head(2):
    doi = fetch_sample_doi(oid)
    if not doi:
        abstract_results[oid] = {'doi': None, 'note': 'DOIなし'}
        continue
    encoded_doi = quote(doi, safe='')
    record, payload = scopus_get(f'Abstract Retrieval: {oid}',
                                  f'https://api.elsevier.com/content/abstract/doi/{encoded_doi}')
    all_call_records.append(record)
    title = None
    citedby = None
    if payload:
        coredata = payload.get('abstracts-retrieval-response', {}).get('coredata', {})
        title = coredata.get('dc:title')
        citedby = coredata.get('citedby-count')
    abstract_results[oid] = {'doi': doi, 'title': title, 'scopus_citedby_count': citedby}
    time.sleep(0.3)

pd.DataFrame(abstract_results).T

,doi,title,scopus_citedby_count
A5112713997,10.1021/jp9828179,Fluorescence and EPR characteristics of Mn2+-d...,301
A5010897098,10.1109/jsen.2004.833152,A tactile sensor sheet using pressure conducti...,421


## 7. 突き合わせ結果のまとめ（OpenAlex vs Scopus）

サンプル各人について、OpenAlex側の情報（`sample_group`＝既知ノイズか現所属か）と、Scopus側から得られた根拠
（Author Searchでの一致有無、Scopus上の現在所属がToyotaか、AFFIL/AF-ID検索でのヒット件数）を横並びにする。
`known_noise`のOpenAlexエントリでScopusの現在所属がToyota以外・非該当が多ければ、Scopusによるノイズ検出が機能している証拠になる。

In [8]:
verdict_rows = []
for _, row in sample.iterrows():
    oid = row['openalex_id']
    res = author_search_results.get(oid, {'candidates': [], 'name_order_used': None})
    cands = res['candidates']
    top = cands[0] if cands else {}
    cmp_row = search_comparison.get(oid, {})
    verdict_rows.append({
        'sample_group': row['sample_group'],
        'name': row['name'],
        'openalex_first_pub_year': row['first_pub_year'],
        'openalex_is_currently_affiliated': row['is_currently_affiliated'],
        'openalex_matched_entities': row['matched_entities'],
        'scopus_author_search_hit': bool(cands),
        'scopus_current_affiliation': top.get('affiliation_current'),
        'scopus_current_affiliation_is_toyota': top.get('affiliation_current_is_toyota'),
        'scopus_broad_affil_hits': cmp_row.get('broad_affil_hits'),
        'scopus_narrow_af_id_hits': cmp_row.get('narrow_af_id_hits'),
    })

verdict_df = pd.DataFrame(verdict_rows)
verdict_df

,sample_group,name,openalex_first_pub_year,openalex_is_currently_affiliated,openalex_matched_entities,scopus_author_search_hit,scopus_current_affiliation,scopus_current_affiliation_is_toyota,scopus_broad_affil_hits,scopus_narrow_af_id_hits
0,known_noise,Tetsuo Yazawa,1936,False,Toyota Motor Corporation (Japan),True,University of Hyogo,False,0,0
1,known_noise,Masatoshi Ishikawa,1942,False,Toyota Motor Corporation (Japan),False,NaN,None,0,0
2,known_noise,Shin Yamamoto,1935,False,Toyota Motor Corporation (Switzerland),True,Aichi Prefectural University,False,10,0
3,known_noise,Minoru KAWAMOTO,1936,False,Toyota Motor Corporation (Switzerland),False,NaN,None,0,0
4,currently_affiliated,Tomohiro MIYABE,2002,True,Toyota Motor Corporation (Switzerland),True,"Toyota Central R&D Labs., Inc.",True,0,0
5,currently_affiliated,Dorothée Lahaussois,2022,True,Toyota Motor Corporation (Belgium),True,Toyota Motor Europe NV/SA,True,0,0
6,currently_affiliated,Masaki Nobuhiro,2022,True,Toyota Motor Corporation (Switzerland); Toyota...,True,IDEC Corporation,False,4,0
7,currently_affiliated,Masahiro Doi,2026,True,Toyota Motor Corporation (Japan),True,Toyota Motor Corporation,True,4,0


## 8. 結果の保存

APIキー自体を含まない、比較・分析用の最小限の記録だけを`data/derived/scopus_pilot_exploration/`に保存する。

In [9]:
call_log_df = pd.DataFrame(all_call_records)
call_log_path = OUTPUT_DIR / 'api_call_log.csv'
call_log_df.to_csv(call_log_path, index=False)

sample_path = OUTPUT_DIR / 'sample_selection.csv'
sample[sample_cols].to_csv(sample_path, index=False)

affiliation_path = OUTPUT_DIR / 'affiliation_search_candidates.csv'
affiliation_df.to_csv(affiliation_path, index=False)

verdict_path = OUTPUT_DIR / 'verdict_comparison.csv'
verdict_df.to_csv(verdict_path, index=False)

summary = {
    'run_at_utc': RUN_AT_UTC,
    'n_api_calls': len(all_call_records),
    'n_errors': int((call_log_df['http_status'].fillna(0).astype(int) >= 400).sum()) if not call_log_df.empty else 0,
    'note': 'Scopus単独でのパイロット探索。WoS Starter APIはClarivate承認待ちのため未実施。',
}
with (OUTPUT_DIR / 'run_summary.json').open('w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f'保存: {call_log_path.relative_to(ROOT)}')
print(f'保存: {sample_path.relative_to(ROOT)}')
print(f'保存: {affiliation_path.relative_to(ROOT)}')
print(f'保存: {verdict_path.relative_to(ROOT)}')
print(summary)
call_log_df

保存: data/derived/scopus_pilot_exploration/api_call_log.csv
保存: data/derived/scopus_pilot_exploration/sample_selection.csv
保存: data/derived/scopus_pilot_exploration/affiliation_search_candidates.csv
保存: data/derived/scopus_pilot_exploration/verdict_comparison.csv
{'run_at_utc': '2026-07-27T01:35:01.188931+00:00', 'n_api_calls': 35, 'n_errors': 0, 'note': 'Scopus単独でのパイロット探索。WoS Starter APIはClarivate承認待ちのため未実施。'}


,source,http_status,duration_ms,rate_limit_remaining,rate_limit_limit,url
0,Affiliation Search: Toyota Motor,200,698,4999,5000,https://api.elsevier.com/content/search/affili...
1,"Author Search: Yazawa,Tetsuo+Toyota",200,360,4996,5000,https://api.elsevier.com/content/search/author
2,"Author Search: Ishikawa,Masatoshi+Toyota",200,505,4995,5000,https://api.elsevier.com/content/search/author
3,"Author Search: Masatoshi,Ishikawa+Toyota",200,497,4994,5000,https://api.elsevier.com/content/search/author
4,"Author Search: Yamamoto,Shin+Toyota",200,487,4993,5000,https://api.elsevier.com/content/search/author
5,"Author Search: KAWAMOTO,Minoru+Toyota",200,353,4992,5000,https://api.elsevier.com/content/search/author
6,"Author Search: Minoru,KAWAMOTO+Toyota",200,396,4991,5000,https://api.elsevier.com/content/search/author
7,"Author Search: MIYABE,Tomohiro+Toyota",200,370,4990,5000,https://api.elsevier.com/content/search/author
8,"Author Search: Lahaussois,Dorothée+Toyota",200,472,4989,5000,https://api.elsevier.com/content/search/author
9,"Author Search: Nobuhiro,Masaki+Toyota",200,377,4988,5000,https://api.elsevier.com/content/search/author
